In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates

In [2]:

cities = {
    # ===== MIỀN BẮC (6 tỉnh) =====
    "Hanoi": (21.0278, 105.8342),
    "Thai Nguyen": (21.5671, 105.8252),      # ⭐ MỚI - Trung du miền núi
    "Vinh Phuc": (21.3609, 105.5474),        # ⭐ MỚI - Vùng đồng bằng
    "Quang Ninh": (21.0064, 107.2925),       # ⭐ MỚI - Hạ Long, vùng biển
    
    # ===== MIỀN TRUNG (6 tỉnh) =====
    "Da Nang": (16.0544, 108.2022),
    "Hue": (16.4637, 107.5909),
    "Quang Nam": (15.5394, 108.0191),        # ⭐ MỚI - Hội An, Tam Kỳ
    "Khanh Hoa": (12.2585, 109.0526),        # ⭐ MỚI - Nha Trang, du lịch biển
    
    # ===== MIỀN NAM (6 tỉnh) =====
    "Ho Chi Minh": (10.7769, 106.7009),
    "Dong Nai": (10.9465, 106.8340),         # ⭐ MỚI - Biên Hòa, công nghiệp
    "Ba Ria Vung Tau": (10.5417, 107.2429),  # ⭐ MỚI - Vũng Tàu, biển
    "Long An": (10.5336, 106.4110),          # ⭐ MỚI - Tân An, đồng bằng sông Cửu Long
}

start_date = "2014-01-01"
end_date   = "2024-12-31"

# Các biến có sẵn trong Archive API
daily_vars = [
    "temperature_2m_max",
    "temperature_2m_min", 
    "temperature_2m_mean",
    "rain_sum",
    "precipitation_hours",
    "weather_code",
    "sunrise",
    "sunset",
    "sunshine_duration",
    "daylight_duration",  
    "wind_speed_10m_max",
    "wind_speed_10m_mean",
    "wind_gusts_10m_max",
    "wind_direction_10m_dominant",
    "shortwave_radiation_sum",
]

records = []

for city, (lat, lon) in cities.items():
    url = (
        "https://archive-api.open-meteo.com/v1/archive?"  # Đúng endpoint
        f"latitude={lat}&longitude={lon}"
        f"&start_date={start_date}&end_date={end_date}"
        f"&daily={','.join(daily_vars)}"
        "&timezone=Asia/Bangkok"  # Timezone rõ ràng
    )

    try:
        print(f"🔄 Đang lấy dữ liệu cho {city}...")
        res = requests.get(url, timeout=30).json()
        
        if "daily" in res:
            daily = res["daily"]
            for i in range(len(daily["time"])):
                record = {"city": city, "latitude": lat, "longitude": lon, "date": daily["time"][i]}
                for var in daily_vars:
                    record[var] = daily.get(var, [None]*len(daily["time"]))[i]
                records.append(record)
            print(f"✅ {city}: {len(daily['time'])} ngày")
        else:
            print(f"⚠️ {city}: {res.get('reason', res)}")
            
        time.sleep(0.5)  # Tránh spam API
        
    except Exception as e:
        print(f"❌ Lỗi {city}: {e}")
        
    time.sleep(30)

df_daily = pd.DataFrame(records)
# df = pd.DataFrame(records)
# print(f"\n{'='*60}")
# print(f"Tổng số record: {len(df)}")
# print(f"\nMẫu dữ liệu:")
# print(df.head(10))

# if len(df) > 0:
#     df.to_csv("vietnam_weather_2023_archive.csv", index=False, encoding="utf-8")
#     print("\n✅ Đã lưu file thành công!")

🔄 Đang lấy dữ liệu cho Hanoi...
✅ Hanoi: 4018 ngày
🔄 Đang lấy dữ liệu cho Thai Nguyen...
✅ Thai Nguyen: 4018 ngày
🔄 Đang lấy dữ liệu cho Vinh Phuc...
✅ Vinh Phuc: 4018 ngày
🔄 Đang lấy dữ liệu cho Quang Ninh...
✅ Quang Ninh: 4018 ngày
🔄 Đang lấy dữ liệu cho Da Nang...
✅ Da Nang: 4018 ngày
🔄 Đang lấy dữ liệu cho Hue...
✅ Hue: 4018 ngày
🔄 Đang lấy dữ liệu cho Quang Nam...
✅ Quang Nam: 4018 ngày
🔄 Đang lấy dữ liệu cho Khanh Hoa...
✅ Khanh Hoa: 4018 ngày
🔄 Đang lấy dữ liệu cho Ho Chi Minh...
✅ Ho Chi Minh: 4018 ngày
🔄 Đang lấy dữ liệu cho Dong Nai...
✅ Dong Nai: 4018 ngày
🔄 Đang lấy dữ liệu cho Ba Ria Vung Tau...
✅ Ba Ria Vung Tau: 4018 ngày
🔄 Đang lấy dữ liệu cho Long An...
✅ Long An: 4018 ngày


In [3]:
hourly_vars = [
    "relative_humidity_2m",   # Độ ẩm
    "dewpoint_2m",            # Điểm sương
    "surface_pressure",       # Áp suất
    "cloudcover",             # Mây
]

hourly_records = []

print("\n" + "=" * 70)
print("📊 BƯỚC 2: Lấy dữ liệu HOURLY (để tính daily)")
print("=" * 70)

for city, (lat, lon) in cities.items():
    url = (
        "https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={lat}&longitude={lon}"
        f"&start_date={start_date}&end_date={end_date}"
        f"&hourly={','.join(hourly_vars)}"
        "&timezone=Asia/Bangkok"
    )
    
    try:
        print(f"🔄 {city}...", end=" ")
        res = requests.get(url, timeout=30).json()
        
        if "hourly" in res:
            hourly = res["hourly"]
            for i in range(len(hourly["time"])):
                record = {
                    "city": city, 
                    "datetime": hourly["time"][i],
                }
                for var in hourly_vars:
                    record[var] = hourly.get(var, [None]*len(hourly["time"]))[i]
                hourly_records.append(record)
            print(f"✅ {len(hourly['time'])} giờ")
        else:
            print(f"⚠️ Lỗi: {res.get('reason', 'Unknown')}")
        
        time.sleep(2)
    except Exception as e:
        print(f"❌ Lỗi: {e}")

df_hourly = pd.DataFrame(hourly_records)

# ========================================
# BƯỚC 3: Tính toán Daily từ Hourly
# ========================================
print("\n" + "=" * 70)
print("📊 BƯỚC 3: Tính toán Daily từ Hourly")
print("=" * 70)

if len(df_hourly) > 0:
    # Tạo cột date từ datetime
    df_hourly['date'] = pd.to_datetime(df_hourly['datetime']).dt.date
    df_hourly['date'] = df_hourly['date'].astype(str)
    
    # Tính toán các chỉ số daily
    df_hourly_agg = df_hourly.groupby(['city', 'date']).agg({
        'relative_humidity_2m': ['max', 'min', 'mean'],
        'dewpoint_2m': ['max', 'min', 'mean'],
        'surface_pressure': 'mean',
        'cloudcover': 'mean',
    }).reset_index()
    
    # Làm phẳng tên cột
    df_hourly_agg.columns = [
        'city', 'date',
        'relative_humidity_2m_max', 'relative_humidity_2m_min', 'relative_humidity_2m_mean',
        'dewpoint_2m_max', 'dewpoint_2m_min', 'dewpoint_2m_mean',
        'surface_pressure_mean',
        'cloudcover_mean',
    ]
    
    print(f"✅ Đã tính toán xong {len(df_hourly_agg)} dòng daily từ hourly")
    
    # ========================================
    # BƯỚC 4: Merge 2 DataFrame
    # ========================================
    print("\n" + "=" * 70)
    print("📊 BƯỚC 4: Gộp dữ liệu Daily + Hourly aggregated")
    print("=" * 70)
    
    df_final = df_daily.merge(
        df_hourly_agg, 
        on=['city', 'date'], 
        how='left'
    )
    
    print(f"✅ Tổng số record: {len(df_final)}")
    print(f"✅ Tổng số cột: {len(df_final.columns)}")
    
    # ========================================
    # BƯỚC 5: Lưu file
    # ========================================
    print("\n" + "=" * 70)
    print("💾 Lưu file CSV")
    print("=" * 70)
    
    df_final.to_csv("../data/raw/vietnam_weather_2023_complete.csv", index=False, encoding="utf-8")
    print("✅ Đã lưu: vietnam_weather_2023_complete.csv")
    
    # Hiển thị thông tin
    print(f"\n📋 Các cột trong file:")
    for i, col in enumerate(df_final.columns, 1):
        print(f"  {i:2d}. {col}")
    
    print(f"\n🔍 Mẫu dữ liệu (5 dòng đầu):")
    print(df_final.head())
    
    print(f"\n📊 Thống kê missing values:")
    missing = df_final.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    if len(missing) > 0:
        print(missing)
    else:
        print("✅ Không có giá trị thiếu!")
        
else:
    print("⚠️ Không có dữ liệu hourly để xử lý")
    df_final = df_daily
    df_final.to_csv("vietnam_weather_2023_complete.csv", index=False, encoding="utf-8")


📊 BƯỚC 2: Lấy dữ liệu HOURLY (để tính daily)
🔄 Hanoi... ✅ 96432 giờ
🔄 Thai Nguyen... ✅ 96432 giờ
🔄 Vinh Phuc... ✅ 96432 giờ
🔄 Quang Ninh... ✅ 96432 giờ
🔄 Da Nang... ✅ 96432 giờ
🔄 Hue... ✅ 96432 giờ
🔄 Quang Nam... ✅ 96432 giờ
🔄 Khanh Hoa... ✅ 96432 giờ
🔄 Ho Chi Minh... ✅ 96432 giờ
🔄 Dong Nai... ✅ 96432 giờ
🔄 Ba Ria Vung Tau... ✅ 96432 giờ
🔄 Long An... ✅ 96432 giờ

📊 BƯỚC 3: Tính toán Daily từ Hourly
✅ Đã tính toán xong 48216 dòng daily từ hourly

📊 BƯỚC 4: Gộp dữ liệu Daily + Hourly aggregated
✅ Tổng số record: 48216
✅ Tổng số cột: 27

💾 Lưu file CSV
✅ Đã lưu: vietnam_weather_2023_complete.csv

📋 Các cột trong file:
   1. city
   2. latitude
   3. longitude
   4. date
   5. temperature_2m_max
   6. temperature_2m_min
   7. temperature_2m_mean
   8. rain_sum
   9. precipitation_hours
  10. weather_code
  11. sunrise
  12. sunset
  13. sunshine_duration
  14. daylight_duration
  15. wind_speed_10m_max
  16. wind_speed_10m_mean
  17. wind_gusts_10m_max
  18. wind_direction_10m_dominant
  